# 04 — ESM Cambrian: ERAP2 Variant Effect Scoring

**Goal:** Use ESM Cambrian (ESM C 600M) to predict the functional impact of ERAP2 mutations.

Compares wild-type ERAP2 (Q6P179) against the rs2549794 variant to quantify how the Black Death
survival mutation alters protein function.

**Run on:** Google Colab (GPU required — A100 preferred, T4 minimum)

## References
- ESM Cambrian: https://huggingface.co/EvolutionaryScale/esmc-600m-2024-12
- ERAP2 UniProt: Q6P179

In [ ]:
# Install dependencies
!pip install -q transformers torch accelerate esm

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load ESM C 600M model
model_name = "EvolutionaryScale/esmc-600m-2024-12"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name).to(device)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

In [ ]:
# ERAP2 wild-type sequence (Q6P179, 960 aa)
# Upload from data/processed/erap2_Q6P179.fasta or paste here
# TODO: Upload FASTA from local pipeline output
ERAP2_WT = ""  # Paste wild-type sequence here after running script 02

# rs2549794 variant — this SNP affects an amino acid change in ERAP2
# The specific residue change needs to be mapped from the genomic coordinate
# to the protein position using UniProt's variant annotation
VARIANT_POS = None  # Will be set after mapping
VARIANT_AA = None   # Mutant amino acid

In [ ]:
def score_mutation(sequence: str, position: int, wt_aa: str, mt_aa: str) -> dict:
    """
    Score the effect of a single amino acid mutation using ESM masked marginals.
    
    Returns log-likelihood ratio (negative = deleterious, positive = beneficial).
    """
    # Mask the target position
    masked_seq = sequence[:position] + tokenizer.mask_token + sequence[position+1:]
    
    inputs = tokenizer(masked_seq, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get log probabilities at the masked position
    # Position in tokenized sequence (account for CLS token)
    mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1].item()
    logits = outputs.logits[0, mask_idx]
    log_probs = torch.log_softmax(logits, dim=-1)
    
    wt_token = tokenizer.convert_tokens_to_ids(wt_aa)
    mt_token = tokenizer.convert_tokens_to_ids(mt_aa)
    
    wt_logprob = log_probs[wt_token].item()
    mt_logprob = log_probs[mt_token].item()
    
    return {
        "position": position,
        "wt_aa": wt_aa,
        "mt_aa": mt_aa,
        "wt_logprob": wt_logprob,
        "mt_logprob": mt_logprob,
        "delta_logprob": mt_logprob - wt_logprob,  # negative = mutation is disfavored
        "effect": "deleterious" if mt_logprob < wt_logprob else "neutral/beneficial",
    }

In [ ]:
# Score the variant (run after setting ERAP2_WT and VARIANT_POS above)
if ERAP2_WT and VARIANT_POS is not None and VARIANT_AA is not None:
    wt_aa = ERAP2_WT[VARIANT_POS]
    result = score_mutation(ERAP2_WT, VARIANT_POS, wt_aa, VARIANT_AA)
    print(f"Mutation: {result['wt_aa']}{result['position']+1}{result['mt_aa']}")
    print(f"WT log-prob: {result['wt_logprob']:.4f}")
    print(f"MT log-prob: {result['mt_logprob']:.4f}")
    print(f"Delta: {result['delta_logprob']:.4f}")
    print(f"Predicted effect: {result['effect']}")
else:
    print("Set ERAP2_WT, VARIANT_POS, and VARIANT_AA first.")
    print("Run scripts 01-03 locally, then upload erap2_Q6P179.fasta.")

In [ ]:
# Scan entire ERAP2 sequence for mutation-sensitive positions
# (identifies structurally/functionally important residues)
if ERAP2_WT:
    print("Scanning ERAP2 for mutation-sensitive positions ...")
    # This is a deep mutational scan — takes ~10-30 min on A100
    # Sample every 10th position for quick check first
    sensitive_positions = []
    for pos in range(0, len(ERAP2_WT), 10):
        wt_aa = ERAP2_WT[pos]
        # Score all possible substitutions
        masked_seq = ERAP2_WT[:pos] + tokenizer.mask_token + ERAP2_WT[pos+1:]
        inputs = tokenizer(masked_seq, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1].item()
        logits = outputs.logits[0, mask_idx]
        log_probs = torch.log_softmax(logits, dim=-1)
        wt_token = tokenizer.convert_tokens_to_ids(wt_aa)
        wt_logprob = log_probs[wt_token].item()
        # High WT log-prob = position is highly conserved
        if wt_logprob > -0.5:
            sensitive_positions.append((pos, wt_aa, wt_logprob))
    print(f"Found {len(sensitive_positions)} highly conserved positions (sampled every 10th)")